In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)


In [2]:
from scipy.stats import mode
from sklearn import datasets, tree
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (roc_auc_score, confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score,
                             classification_report)

In [3]:


from imblearn.over_sampling import SMOTE
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.utils import concordance_index



In [4]:
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
import pandas as pd

data =pd.read_csv("final_dataset_PART2.csv")

print(f'Dataset Shape: {data.shape}')
print(f' Dataset Memory Usage: {data.memory_usage().sum() / 1024 ** 2:.2f} MB')

Dataset Shape: (1480, 62)
 Dataset Memory Usage: 0.32 MB


In [6]:
cancer_type = data['event'].value_counts()
print(cancer_type)

event
0    835
1    645
Name: count, dtype: int64


In [7]:
#Vyvorenie premennej x a y, pricom x su predikujuce atributu a y je predikovany atribut GRADE
x = data.drop(['event'], axis = 1)
y = data['event']


In [8]:
#Rozdelenie mnoziny stratifikovane na trenovaciu a testovaciu 

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.30, stratify=y)
print(f"Pôvodný počet vzoriek (trénovací set): {y_train.value_counts()}")


Pôvodný počet vzoriek (trénovací set): event
0    584
1    452
Name: count, dtype: int64


In [9]:
#Vyvorenie a natrenovanie modelu 
from sklearn.ensemble import RandomForestClassifier
model1 = RandomForestClassifier()
model1.fit(X_train, y_train)

RandomForestClassifier()

In [10]:
#Vytvorenie premenej testPred_tree1, ktora obsahuje predikovane hodnoty
testPred_tree1 = model1.predict(X_test)

In [11]:
#Vypis kontingencnej tabulky
table = pd.crosstab(testPred_tree1, y_test)
table

event,0,1
row_0,,
0,240,10
1,11,183


In [12]:
#vypis klasifikacneho reportu
from sklearn.metrics import classification_report
print(classification_report(y_test, testPred_tree1))

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       251
           1       0.94      0.95      0.95       193

    accuracy                           0.95       444
   macro avg       0.95      0.95      0.95       444
weighted avg       0.95      0.95      0.95       444



In [13]:
accuracy = model1.score(X_test, y_test)
print(f"Celková accuracy modelu: {accuracy:.2f}")

Celková accuracy modelu: 0.95


### SMOTE OVERSAMPLING
##### ako predikčné atribúty použijeme všetky

In [14]:
# Vytvorenie premennej x a y, pričom x sú predikujúce atribúty a y je predikovaný atribút 'event2'
x = data.drop(['event'], axis=1)
y = data['event']

In [15]:
# Rozdelenie množiny stratifikovane na tréningovú a testovaciu
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.30, stratify=y, random_state=42)

print(f"Pôvodný počet vzoriek (trénovací set): {y_train.value_counts()}")

Pôvodný počet vzoriek (trénovací set): event
0    584
1    452
Name: count, dtype: int64


In [16]:
# Vyváženie tried pomocou SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"Po oversamplingu počet vzoriek (vyvážený trénovací set): {y_resampled.value_counts()}")


Po oversamplingu počet vzoriek (vyvážený trénovací set): event
0    584
1    584
Name: count, dtype: int64


In [17]:
# Vytvorenie a natrénovanie modelu
model2 = RandomForestClassifier(random_state=42)
model2.fit(X_resampled, y_resampled)

# Vytvorenie premennej testPred_tree2, ktorá obsahuje predikované hodnoty
testPred_tree2 = model2.predict(X_test)

# Vypis kontingenčnej tabulky
table = pd.crosstab(testPred_tree2, y_test)
print("Kontingenčná tabuľka:")
print(table)

# Vypis klasifikačného reportu
print("Klasifikačný report:")
print(classification_report(y_test, testPred_tree2))

Kontingenčná tabuľka:
event    0    1
row_0          
0      233   11
1       18  182
Klasifikačný report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       251
           1       0.91      0.94      0.93       193

    accuracy                           0.93       444
   macro avg       0.93      0.94      0.93       444
weighted avg       0.94      0.93      0.93       444



In [18]:
accuracy = model2.score(X_test, y_test)
print(f"Celková accuracy modelu s použitím oversamplingu: {accuracy:.2f}")

Celková accuracy modelu s použitím oversamplingu: 0.93


### SMOTE Undersampling

In [19]:
# Vytvorenie premennej x a y, pričom x sú predikujúce atribúty a y je predikovaný atribút 'event2'
x = data.drop(['event'], axis=1)
y = data['event']


In [20]:
# Rozdelenie množiny stratifikovane na tréningovú a testovaciu
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.30, stratify=y, random_state=42)

print(f"Pôvodný počet vzoriek (trénovací set): {y_train.value_counts()}")

Pôvodný počet vzoriek (trénovací set): event
0    584
1    452
Name: count, dtype: int64


In [21]:
# Undersampling pomocou RandomUnderSampler
from imblearn.under_sampling import RandomUnderSampler
under = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under.fit_resample(X_train, y_train)
print(f"Po oversamplingu počet vzoriek (vyvážený trénovací set): {y_train_under.value_counts()}")


Po oversamplingu počet vzoriek (vyvážený trénovací set): event
0    452
1    452
Name: count, dtype: int64


In [22]:
# Vytvorenie a natrenovanie modelu s undersampling
model3 = RandomForestClassifier()
model3.fit(X_train_under, y_train_under)

# Vytvorenie premenej testPred_tree_under3, ktora obsahuje predikovane hodnoty
testPred_tree_under3 = model3.predict(X_test)

# Vypis kontingencnej tabulky po undersampling
table_under = pd.crosstab(testPred_tree_under3, y_test)
print("Kontingenčná tabuľka po undersampling:")
print(table_under)

Kontingenčná tabuľka po undersampling:
event    0    1
row_0          
0      227    3
1       24  190


In [23]:
# Vypis klasifikačného reportu
print("Klasifikačný report:")
print(classification_report(y_test, testPred_tree_under3))

Klasifikačný report:
              precision    recall  f1-score   support

           0       0.99      0.90      0.94       251
           1       0.89      0.98      0.93       193

    accuracy                           0.94       444
   macro avg       0.94      0.94      0.94       444
weighted avg       0.94      0.94      0.94       444



In [24]:
accuracy = model3.score(X_test, y_test)
print(f"Celková accuracy modelu s použitím undersamplingu: {accuracy:.2f}")

Celková accuracy modelu s použitím undersamplingu: 0.94


### Teray urobíme ten istý postup, len na modelovanie použijeme len atribúty, ktoré sme použili v Cox regresii

In [25]:
relevant_attributes = [
    'event',
    'time',
    'Age at Diagnosis',
    'Type of Breast Surgery',
    'ER status measured by IHC',
    'Nottingham prognostic index',
    'PR Status',
        'Relapse Free Status (Months)', 


]

In [27]:
# Výber iba relevantných atribútov z datasetu
from sklearn.metrics import accuracy_score

data_relevant = data[relevant_attributes]

# Vyvorenie premennej x a y, kde x sú predikujúce atribúty a y je predikovaný atribút
x = data_relevant.drop(['event'], axis=1)
y = data_relevant['event']

# Rozdelenie dát
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.30, stratify=y)

# Vytvorenie a natrenovanie modelu bez undersampling
model4 = RandomForestClassifier()
model4.fit(X_train, y_train)

# Vytvorenie premenej testPred_tree4, ktora obsahuje predikovane hodnoty
testPred_tree4 = model4.predict(X_test)

# Vypis kontingencnej tabulky
table = pd.crosstab(testPred_tree4, y_test)
print("Kontingenčná tabuľka:")
print(table)

# Vypocet a vypis accuracy modelu
accuracy = accuracy_score(y_test, testPred_tree4)
print(f"Celková accuracy modelu: {accuracy:.2f}")

Kontingenčná tabuľka:
event    0    1
row_0          
0      219   47
1       31  147
Celková accuracy modelu: 0.82


In [28]:
# Vypis klasifikačného reportu
print("Klasifikačný report:")
print(classification_report(y_test, testPred_tree4))

Klasifikačný report:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       250
           1       0.83      0.76      0.79       194

    accuracy                           0.82       444
   macro avg       0.82      0.82      0.82       444
weighted avg       0.82      0.82      0.82       444



#### Relevant attributes + oversampling 

In [29]:
# Výber iba relevantných atribútov z datasetu
data_relevant = data[relevant_attributes]

# Vyvorenie premennej x a y, kde x sú predikujúce atribúty a y je predikovaný atribút
x = data_relevant.drop(['event'], axis=1)
y = data_relevant['event']

# Rozdelenie dát
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.30, stratify=y)

# Oversampling pomocou SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

# Vytvorenie a natrenovanie modelu s oversampling
model5 = RandomForestClassifier()
model5.fit(X_resampled, y_resampled)

# Vytvorenie premenej testPred_tree5, ktora obsahuje predikovane hodnoty
testPred_tree5 = model5.predict(X_test)

# Vypis kontingencnej tabulky
table = pd.crosstab(testPred_tree5, y_test)
print("Kontingenčná tabuľka po oversampling:")
print(table)

# Vypocet a vypis accuracy modelu
accuracy = accuracy_score(y_test, testPred_tree5)
print(f"Celková accuracy modelu: {accuracy:.2f}")

Kontingenčná tabuľka po oversampling:
event    0    1
row_0          
0      220   38
1       31  155
Celková accuracy modelu: 0.84


In [30]:
# Vypis klasifikačného reportu
print("Klasifikačný report:")
print(classification_report(y_test, testPred_tree5))

Klasifikačný report:
              precision    recall  f1-score   support

           0       0.85      0.88      0.86       251
           1       0.83      0.80      0.82       193

    accuracy                           0.84       444
   macro avg       0.84      0.84      0.84       444
weighted avg       0.84      0.84      0.84       444



#### Relevant attributes + undersampling

In [31]:
# Výber iba relevantných atribútov z datasetu
data_relevant = data[relevant_attributes]

# Vyvorenie premennej x a y, kde x sú predikujúce atribúty a y je predikovaný atribút
x = data_relevant.drop(['event'], axis=1)
y = data_relevant['event']

# Rozdelenie dát
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.20, stratify=y)

# Undersampling pomocou RandomUnderSampler
under = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = under.fit_resample(X_train, y_train)

# Vytvorenie a natrenovanie modelu s undersampling
model6 = RandomForestClassifier()
model6.fit(X_train_under, y_train_under)

# Vytvorenie premenej testPred_tree6, ktora obsahuje predikovane hodnoty
testPred_tree6 = model6.predict(X_test)

# Vypis kontingencnej tabulky
table = pd.crosstab(testPred_tree6, y_test)
print("Kontingenčná tabuľka po undersampling:")
print(table)

# Vypocet a vypis accuracy modelu
accuracy = accuracy_score(y_test, testPred_tree6)
print(f"Celková accuracy modelu: {accuracy:.2f}")

Kontingenčná tabuľka po undersampling:
event    0    1
row_0          
0      139   22
1       28  107
Celková accuracy modelu: 0.83


In [32]:
# Vypis klasifikačného reportu
print("Klasifikačný report:")
print(classification_report(y_test, testPred_tree6))

Klasifikačný report:
              precision    recall  f1-score   support

           0       0.86      0.83      0.85       167
           1       0.79      0.83      0.81       129

    accuracy                           0.83       296
   macro avg       0.83      0.83      0.83       296
weighted avg       0.83      0.83      0.83       296



### FORWARD STEPWISE SELECTION
##### nemaju tieto atributy nejaky vyrazny vplyv na zlepsenie modelu

In [32]:
from sklearn.linear_model import LogisticRegression
# Funkcia na forward stepwise selection
def forward_stepwise_selection(X, y):
    # Začneme s prázdnym modelom
    selected_features = []
    remaining_features = list(X.columns)
    best_accuracy = 0

    # Pokračujeme, pokiaľ sú stále atribúty, ktoré môžeme pridať
    while remaining_features:
        accuracies = []
        # Pre každý atribút v zozname zostávajúcich atribútov
        for feature in remaining_features:
            # Vytvoríme nový model s aktuálnymi vybranými atribútmi a týmto novým atribútom
            model = LogisticRegression(max_iter=10000)
            current_features = selected_features + [feature]
            X_train, X_test, y_train, y_test = train_test_split(X[current_features], y, test_size=0.2, random_state=42)

            # Trénujeme model a vypočítame accuracy
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            accuracies.append((accuracy, feature))

        # Vyberieme atribút s najvyššou accuracy
        best_accuracy_new, best_feature = max(accuracies, key=lambda x: x[0])

        # Ak nová accuracy zlepší predchádzajúcu accuracy, pridáme tento atribút do modelu
        if best_accuracy_new > best_accuracy:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            best_accuracy = best_accuracy_new
        else:
            break  # Ak pridanie atribútu nezlepší model, ukončíme výber

    return selected_features

# Použitie funkcie
data =pd.read_csv("final_dataset_PART2.csv")
X = data.drop("event", axis=1)  # 'event2' je predikovaný atribút
y = data["event"]

# Zavolanie forward stepwise selection
selected_features = forward_stepwise_selection(X, y)

# Výpis vybraných atribútov
print("Vybrané atribúty:", selected_features)


Vybrané atribúty: ['Relapse Free Status', 'Age at Diagnosis', 'Cohort', 'time', 'Tumor Stage', 'IC_7', 'HER2_Status_Loss', '3-GCS_ER-/HER2-']


In [92]:
# Definovanie relevantných atribútov
relevant_attributes2 = [
    'event',
    'Relapse Free Status', 'Age at Diagnosis', 'Cohort', 'time', 'Tumor Stage', 'IC_7', 'HER2_Status_Loss', '3-GCS_ER-/HER2-'
]
# Výber iba relevantných atribútov z datasetu
data_relevant = data[relevant_attributes2]

# Vyvorenie premennej x a y, kde x sú predikujúce atribúty a y je predikovaný atribút
x = data_relevant.drop(['event'], axis=1)
y = data_relevant['event']

# Rozdelenie dát
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.30, stratify=y)

# Vytvorenie a natrenovanie modelu bez undersampling
model7 = RandomForestClassifier()
model7.fit(X_train, y_train)

# Vytvorenie premenej testPred_tree7, ktora obsahuje predikovane hodnoty
testPred_tree7 = model7.predict(X_test)

# Vypis kontingencnej tabulky
table = pd.crosstab(testPred_tree7, y_test)
print("Kontingenčná tabuľka:")
print(table)

# Vypocet a vypis accuracy modelu
accuracy = accuracy_score(y_test, testPred_tree7)
print(f"Celková accuracy modelu: {accuracy:.2f}")

Kontingenčná tabuľka:
event    0    1
row_0          
0      243   15
1        7  179
Celková accuracy modelu: 0.95


In [93]:
# Vypis klasifikačného reportu
print("Klasifikačný report:")
print(classification_report(y_test, testPred_tree7))

Klasifikačný report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96       250
           1       0.96      0.92      0.94       194

    accuracy                           0.95       444
   macro avg       0.95      0.95      0.95       444
weighted avg       0.95      0.95      0.95       444

